### Data Ingestion

In [32]:
### document datastructure

from langchain_core.documents import Document



In [33]:
doc = Document(
    page_content = "This is the main text content I am using to create RAG",
    metadata = {
        "source": "example.txt",
        "pages": 1,
        "author": "Manohar",
        "date_created": "2025-01-01"
    }
)

doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Manohar', 'date_created': '2025-01-01'}, page_content='This is the main text content I am using to create RAG')

In [35]:
# Create a simple txt file
import os
os.makedirs("../data/text_files", exist_ok=True)


In [36]:
sample_texts = {
    "../data/text_files/python_intro.txt": """
    python is a high-level, interpreted language.
    Key Features:
    - Easy to learn and use
    - Extensive Library Support
    """
}

In [37]:
for filepath, content in sample_texts.items():
    with open(filepath, 'w', encoding="utf-8") as f:
        f.write(content)

In [38]:
### TextLoader

# from langchain.document_loaders import TextLoader
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python_intro.txt", encoding="utf-8")
document = loader.load()
print(document)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='\n    python is a high-level, interpreted language.\n    Key Features:\n    - Easy to learn and use\n    - Extensive Library Support\n    ')]


In [39]:
# One more way is using Directory Loader

from langchain_community.document_loaders import DirectoryLoader


dir_loader = DirectoryLoader(
    "../data/text_files", 
    glob = "**/*.txt",
    loader_cls = TextLoader,
    loader_kwargs = {'encoding': 'utf-8'},
    show_progress = False
)

documents = dir_loader.load()
documents

[Document(metadata={'source': '../data/text_files/machineLe.txt'}, page_content='Machine Learning is a Branch of AI\nUsed to create models and all these\n'),
 Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='\n    python is a high-level, interpreted language.\n    Key Features:\n    - Easy to learn and use\n    - Extensive Library Support\n    ')]

In [40]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

dir_loader = DirectoryLoader(
    "../data/text_files", 
    glob = "**/*.pdf",
    loader_cls = PyMuPDFLoader,
    show_progress = False
)

documents = dir_loader.load()
documents

[]

## RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [47]:
import os 
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [54]:
## Read all the pdfs inside the directory

def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print("pdf_files: ", pdf_files)
    print(f"Found {len(pdf_files)} PDF files to process")
    for pdf_file in pdf_files:
        print("each pdf file: ", pdf_file)
        print(f" Processing {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type']='pdf'

            all_documents.extend(documents)
            print("length of documets", len(documents), " pages")
        except Exception as e:
            print("Error :", e)
    return all_documents

all_pdf_documents = process_all_pdfs("../data")

Ignoring wrong pointing object 5 0 (offset 0)
Ignoring wrong pointing object 9 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 19 0 (offset 0)


pdf_files:  [PosixPath('../data/pdf_files/yrlmanoharreddy.pdf'), PosixPath('../data/pdf_files/MarkKula-Section1.pdf'), PosixPath('../data/pdf_files/yrlmanohar_resume.pdf')]
Found 3 PDF files to process
each pdf file:  ../data/pdf_files/yrlmanoharreddy.pdf
 Processing yrlmanoharreddy.pdf
length of documets 1  pages
each pdf file:  ../data/pdf_files/MarkKula-Section1.pdf
 Processing MarkKula-Section1.pdf
length of documets 10  pages
each pdf file:  ../data/pdf_files/yrlmanohar_resume.pdf
 Processing yrlmanohar_resume.pdf
length of documets 2  pages


In [55]:
## Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function=len,
        separators = ["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    if split_docs:
       print(f"\nExample chunk: ")
       print(f"Content: {split_docs[0].page_content[:200]}...")
       print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [56]:
chunks = split_documents(all_pdf_documents)
print(chunks)

Split 13 documents into 37 chunks

Example chunk: 
Content: Case Study 1. 
 
1.1) Mike haven’t fullfilled his daughter’s dream to study in the college for the fall So it would be emotional distress as a result 
of financial loss. Mike lost his time in saving m...
Metadata: {'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2024-01-24T09:57:29-08:00', 'author': 'Meda Yenkatarajalaxmimanohar', 'moddate': '2024-01-24T09:57:29-08:00', 'source': '../data/pdf_files/yrlmanoharreddy.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'yrlmanoharreddy.pdf', 'file_type': 'pdf'}
[Document(metadata={'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2024-01-24T09:57:29-08:00', 'author': 'Meda Yenkatarajalaxmimanohar', 'moddate': '2024-01-24T09:57:29-08:00', 'source': '../data/pdf_files/yrlmanoharreddy.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'yrlmanoharreddy.pdf', 'file_type': 'pdf'}, page_content='Case Study 1. \n \n1.1

### Embedding And VectorStoreDB

In [68]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
class EmbeddingManager:
    """Handles document embedding generation using
    SentenceTransformer
    """
    def __init__(self, model_name: str = "all_miniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace Model name for sentence Embeddings
        """
        self.model_name = model_name
        self.model = None
        self.load_model()

    def _load_model(self):
        """Load the sentenceTransformer model"""
        try:
            print(f"Loading embedding model: "{self.model_name})
            self.model = sentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimensions: " {
                self.model.get_sentence_embedding_dimension()
            })
        except Exception as e:
            print(f"Error loading the model {self.model_name}: {e}")
            raise
    def generate_embedding(self, texts: List[str])->np.array:
        """
        Generate Embeddings for a list of texts

        Args:
        texts: List of text strings to embed

        Returns:
        numpy arrray of embedding
        """
